In [1]:
from ipykernel import get_connection_file
print(get_connection_file())

%load_ext autoreload
%autoreload 2
    
import os
import sys
import yaml

sys.path.append('/home/lishengping/projects/maxtext/MaxText')
# os.environ['HARDWARE'] = 'cpu'

import pyconfig
from layers import models
import max_utils
import jax
import orbax
import jax.numpy as jnp
from jax.sharding import Mesh
from flax.traverse_util import flatten_dict, unflatten_dict
from flax import linen as nn

# # class DreamMiniXLE64T4Align(DreamMiniXLE64T4):
# #     # 配置文件需要更改的几个地方：
# #     base_output_directory = 'gs://newproject-1-llm_base_models_europe-west4'
# #     run_name = 'test'
# #     query_chunk_size = None # 如果传了这个参数，forward需要是query_chunk_size的整数倍
# #     attention = 'dot_product_chunk'
# #     # exp_class set your model class
# #     per_device_batch_size = 1 # 可以根据测试的batch size定，设小一点主要是为了节省显存
# #     max_target_length = 4096 # 可以根据测试的长度定，设小一点主要是为了节省显存
# #     zero_loss = True
# #      # 因为base.yml默认为空，必须写一个
# #     scan_layers = False # 因为转模型的时候转的是scan_layers=False
# #     record_internal_nn_metrics = 0
# #     load_balance_loss_weight = None # dropless moe
# #     megablox = False
# #     bucket_logging_enabled = False
# #     train_stage = 4
    
run_name = 'align'
os.makedirs(run_name, exist_ok=True) # 因为如果不存在会报错
config_name = '/home/lishengping/projects/maxtext/MaxText/configs/base.yml'
argv = [None, config_name]
config = pyconfig.initialize(argv)

/home/lishengping/.local/share/jupyter/runtime/kernel-bd62481d-991a-407c-8975-2fb33fea57be.json


2026-01-29 14:42:46.174600: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769697766.188027  208749 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769697766.191826  208749 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769697766.202659  208749 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697766.202669  208749 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697766.202672  208749 computation_placer.cc:177] computation placer alr

Updating keys from env and command line: []
Running Model: default
Updating keys from model: []
Attempting to initialize the jax distributed system...
Jax distributed system initialized!


Updated exp model vars:
[EXP] adam_b1: 0.9
[EXP] adam_b2: 0.95
[EXP] adam_eps: 1e-08
[EXP] adam_weight_decay: 0.1
[EXP] async_checkpointing: True
[EXP] attention: flash
[EXP] base_emb_dim: 2048
[EXP] base_mlp_dim: 2560
[EXP] base_num_decoder_layers: 32
[EXP] base_num_kv_heads: [32, 4, 32, 32]
[EXP] base_num_query_heads: 32
[EXP] base_output_directory: gs://newproject-1-llm_base_models_us-east5/v4.5-1.5B
[EXP] bucket_logging_enabled: False
[EXP] checkpoint_period: 250
[EXP] compose_layers: range(1, 60, 2)
[EXP] cosine_learning_rate_final_fraction: 0.1
[EXP] data_shuffle_seed: 9876
[EXP] dataset_type: v4.5_1.5B
[EXP] dc_share_prepost_dw_hidden: True
[EXP] dc_use_muon: True
[EXP] ddw_gen_chunk_size: None
[EXP] ddw_gen_pattern: q,k,v,m
[EXP] debug: False
[EXP] decoder_block: fusion
[EXP] deep_embed_init:

2026-01-29 14:42:49.781063: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Not using emergency checkpoint, ignoring local_checkpoint_directory, local_checkpoint_period, use_replicator_service and replicator_backup_interval_minutes
Config param activations_in_float32: False
Config param adam_b1: 0.9
Config param adam_b2: 0.95
Config param adam_eps: 1e-08
Config param adam_eps_root: 0.0
Config param adam_weight_decay: 0.1
Config param add_bos: True
Config param add_eos: True
Config param allow_split_physical_axes: False
Config param ar_cache_axis_order: 1,2,0,3
Config param async_checkpointing: True
Config param attention: flash
Config param attention_type: global
Config param attn_logits_soft_cap: None
Config param autoregressive_decode_assert: 
Config param base_emb_dim: 2048
Config param base_mlp_dim: 2560
Config param base_moe_mlp_dim: 7168
Config param base_num_decoder_layers: 32
Config param base_num_kv_heads: [32, 4, 32, 32]
Config param base_num_query_heads: 32
Config param base_output_directory: gs://newproject-1-llm_base_models_us-east5/v4.5-1.5B
Conf

In [2]:
shapedtype = {}
# for line in open('model_params.txt', 'r'):
for line in open('model_params_newest_0123.txt', 'r'):
    if not line.strip(): continue
    name = line.strip().split(' ')[0]
    shape = line.strip().split(':')[-1]
    shapedtype[name] = eval(shape)
    if 'layers_0/' in name:
        print(name, shape)

params/decoder/layers_0/block/mlp/wi_0/kernel  (2048, 1, 2560)
params/decoder/layers_0/block/mlp/wi_1/kernel  (2048, 1, 2560)
params/decoder/layers_0/block/mlp/wo/kernel  (2560, 1, 2048)
params/decoder/layers_0/block/post_self_attention_layer_norm/scale  (2048, 1)
params/decoder/layers_0/block/pre_self_attention_layer_norm/scale  (2048, 1)
params/decoder/layers_0/block/self_attention/attention_op/q_dyn_w_proj/dd/kernel  (2048, 1, 64)
params/decoder/layers_0/block/self_attention/attention_op/q_dyn_w_proj/dw1/kernel  (2048, 1, 256)
params/decoder/layers_0/block/self_attention/attention_op/q_dyn_w_proj/qkw  (2048, 1, 32)
params/decoder/layers_0/block/self_attention/attention_op/q_dyn_w_proj/w1_bias  (2, 1, 2, 32)
params/decoder/layers_0/block/self_attention/attention_op/q_dyn_w_proj/w2_bias  (2, 1, 2, 32)
params/decoder/layers_0/block/self_attention/key/kernel  (2048, 1, 2048)
params/decoder/layers_0/block/self_attention/kv_shift/kv_shift_proj_k/kernel  (2048, 1, 32)
params/decoder/layers

In [3]:
# pip uninstall torch -y
# pip cache purge
# pip install torch --extra-index-url https://download.pytorch.org/whl/cpu
import torch
import numpy as np
import jax.numpy as jnp

# os.environ['HARDWARE'] = 'cpu'

import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64


import json
    
# load
mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
axes = [1] * len(mesh_axes)
axes[2] = 1
devices = np.asarray(jax.devices()).reshape(axes)
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = jnp.bfloat16 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
for k, shape in shapedtype.items():
    if not isinstance(k, tuple):
        k = tuple(k.split('/'))
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)    
    
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)
# checkpoint_dir = '/home/lishengping/160000_jax/items'
checkpoint_dir = '/home/lishengping/189250/items'

ckpt = epath.Path(checkpoint_dir)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)
# 如果restored只是一个带有模型名字的字典，没有具体的value矩阵，可以检查下abstract_unboxed_params是不是多了或者漏了params这个key
restored = ckptr.restore(
  ckpt, item={'params': abstract_unboxed_params}, transforms={}, restore_args={'params': restore_args}
)


('params', 'token_embedder', 'embedding') (100352, 2048)
('params', 'decoder', 'mudd_embedder', 'embedding') (100352, 40960)
('params', 'decoder', 'layers_0', 'block', 'mlp', 'wi_0', 'kernel') (2048, 1, 2560)
('params', 'decoder', 'layers_0', 'block', 'mlp', 'wi_1', 'kernel') (2048, 1, 2560)
('params', 'decoder', 'layers_0', 'block', 'mlp', 'wo', 'kernel') (2560, 1, 2048)
('params', 'decoder', 'layers_0', 'block', 'post_self_attention_layer_norm', 'scale') (2048, 1)
('params', 'decoder', 'layers_0', 'block', 'pre_self_attention_layer_norm', 'scale') (2048, 1)
('params', 'decoder', 'layers_0', 'block', 'self_attention', 'attention_op', 'q_dyn_w_proj', 'dd', 'kernel') (2048, 1, 64)
('params', 'decoder', 'layers_0', 'block', 'self_attention', 'attention_op', 'q_dyn_w_proj', 'dw1', 'kernel') (2048, 1, 256)
('params', 'decoder', 'layers_0', 'block', 'self_attention', 'attention_op', 'q_dyn_w_proj', 'qkw') (2048, 1, 32)
('params', 'decoder', 'layers_0', 'block', 'self_attention', 'attention_

In [4]:
# torch_params = torch.save(torch_params, 'torch_params.bin')
def model_init(model, config, key):
  input_shape = (config.global_batch_size_to_load, config.max_target_length)
  params = model.init(
      {"params": key, "dropout": key, "aqt": key},
      jnp.ones(input_shape, dtype=jnp.int32),
      jnp.ones(input_shape, dtype=jnp.int32),
  )
  return params

quant = None
# devices_array = max_utils.create_device_mesh(config)
# mesh = Mesh(devices_array, config.mesh_axes)
Transformer = models.Transformer
jax_model = Transformer(config, mesh, quant=quant)

is_train = False
rng1, aqt_rng = jax.random.split(jax.random.key(9876))

In [7]:
import os
import time
import argparse
import socket
import random
from collections import defaultdict
os.environ["JAX_PLATFORMS"] = "cpu"

import tensorflow as tf
import jax
import numpy as np


def _parse_function(example_proto):
    feature_desc = {key: tf.io.VarLenFeature(tf.int64) for key in task_features}
    example = tf.io.parse_single_example(example_proto, feature_desc)
    for name in list(example.keys()):
        t = example[name]
        if t.dtype == tf.int64:
            t = tf.cast(t, dtype=tf.int32)
        example[name] = tf.sparse.to_dense(t, default_value=0)[: seq_len]
        print(f'example[name]: {example[name]}')
    return example

task_features = {'input_ids': None}
train_seed = 1234
num_infeed_hosts = 1
shuffle_buffer_size = None
pad_id = 0
batch_size = 1
seq_len = 4097

fname = ['gs://newproject-1-llm_base_models_us-east5/data/v4.5-1.5B/mp_shuffle_data/validation/000000.tfrecord']
tf.random.set_seed(train_seed)
ds = tf.data.Dataset.from_tensor_slices(fname)
ds = ds.apply(tf.data.TFRecordDataset)
ds = ds.shard(num_infeed_hosts, 0)
ds = ds.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
if shuffle_buffer_size is not None:
    ds = ds.shuffle(buffer_size=shuffle_buffer_size)
padded_shapes = {key: seq_len for key in task_features}
padding_values = {key: pad_id for key in task_features}
ds = ds.padded_batch(
    batch_size=np.prod(batch_size),
    padded_shapes=padded_shapes,
    padding_values=padding_values,
    drop_remainder=True,
)
iter_ds = ds.as_numpy_iterator()
ori_input_ids = next(iter_ds)['input_ids'].tolist()
input_ids = [ori_input_ids[0][:2049]]

example[name]: Tensor("strided_slice:0", shape=(None,), dtype=int32)


In [8]:
# input_ids = [[100257,   3896,    382,    425,    287,   8396,  10368,    320,
#            560,  65221,     11,  11782,  44064,   7422,     11,  18845,
#             11,  79482,     11,   4325,   2191,     11,  63334,    696,
#            791,   9395,    315,    279,  10368,    374,    311,    636,
#           1274,   7422,  93396,    323,   8678,     83,   2740,    382,
#            791,   3917,    374,   1268,    279,   7859,  21801,    315,
#           9191,   1274,    304,   8396,    690,   2349,    279,   1917,
#             11,    719,   3604,    279,   3917,    649,    387,    922,
#            904,   3544,  13230,   9327,   4286,    644,    279,   1162,
#            315,    459,  80043,   8396,   1521,   4442,    527,   2736,
#           5304,    603,     11,    779,    433,    596,    539,    264,
#          59159,  10368,     13,    578,   5820,  14224,   1101,  37167,
#           1274,    311,   1781,    922,   4325,   2191,    323,   4325,
#          22526,   4819,    382,  48614,    750,   2610,   1912,   3697,
#            311,   2980,    323,  10491,   1148,    814,   4510,    690,
#            387,    279,   3254,  12474,   2515,    304,    279,   1828,
#            220,     16,     14,     17,     14,     18,     14,     20,
#           1667,    315,    279,  80043,   7187,    389,    872,   3158,
#            315,   5820,  33526,   2805,   3225,   3262,  55660,  42761,
#            482,    477,    389,   8396,   8965,    482,    320,  42820,
#            323,   3158,    315,   5536,   6773,    555,    279,  17028,
#            859,     11,  11911,    389,    279,  12034,  33526,   2805,
#          13757,    315,    279,   1912,   3677,    791,   6325,    315,
#            279,   1912,   3697,    649,    387,  14407,    477,  10666,
#            477,  59674,  11911,    389,    279,  17028,    859,    596,
#          22262,    323,  17413,    315,    279,   3882,    382,  19997,
#           3585,    649,   2997,   1473,    220,   7436,  22498,   1912,
#           5597,    439,    311,    279,   1455,    824,  59374,  24710,
#            198,    220,   7436,   1148,  18726,    527,    279,   1455,
#          87490,    323,   4741,     12,  66154,    198,    220,   7436,
#           1268,   2204,  18726,   2643,   5536,    389,   1855,   1023,
#            198,    220,   7436,   1148,   2038,    374,  32161,    369, 810]]
import pickle

losses = []
for i in range(1):
    # input_ids = pickle.load(open(f'input_ids{i}.pkl', 'rb'))
    # input_ids = pickle.load(open(f'input_ids{i}.pkl', 'rb'))
    
    batch_size = 1
    input_ids = jnp.array(input_ids).reshape(batch_size, -1)
    data = {}
    data['inputs'] = input_ids[:, :-1]
    labels = input_ids[:, 1:]
    data["inputs_position"] = jnp.arange(data['inputs'].shape[1]).reshape(batch_size, -1)
    data["inputs_segmentation"] = jnp.ones_like(data['inputs'])
    data["targets"] = jnp.array(labels).reshape(batch_size, -1)
    data['targets_segmentation'] = jnp.ones_like(data['inputs'])
    
    params = restored['params']['params']
    outputs, intermediate_outputs = jax_model.apply(
              {'params': params},
              data["inputs"],
              data["inputs_position"],
              decoder_segment_ids=data["inputs_segmentation"],
              decoder_target_mask=data['targets_segmentation'],
              decoder_target_tokens=data["targets"],
              enable_dropout=config.enable_dropout if is_train else False,
              rngs={"dropout": rng1, "params": aqt_rng},
              mutable="intermediates",
          )
    (xent, correct, preds, logits, mtp_logits) = outputs
    loss = round(xent.mean().item(), 4)
    losses.append(loss)
    print(f'{i}-loss: {loss}========================')
    # flat_intermediate_outputs = {'/'.join(k):v for k, v in  flatten_dict(intermediate_outputs).items()}

tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048, 128) value: (1, 16, 2048, 128)
tpu flash query: (1, 16, 2048, 128) key: (1, 16, 2048,